# Model Evaluation - Municipality of Sampaloc Quezon Chatbot
Test model performance on citizen charter services

In [1]:
import json
import pickle
import torch
import faiss
from transformers import AutoTokenizer, AutoModelForSequenceClassification, T5TokenizerFast, T5ForConditionalGeneration
from sentence_transformers import SentenceTransformer
import pandas as pd

c:\Users\stsim\OneDrive\Desktop\GOVERNMENT CHATBOT\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Models

In [2]:
with open('models/time_waster_tfidf.pkl', 'rb') as f:
    tfidf = pickle.load(f)
with open('models/time_waster_classifier.pkl', 'rb') as f:
    tw_classifier = pickle.load(f)

sentiment_tokenizer = AutoTokenizer.from_pretrained('models/sentiment_tokenizer')
sentiment_model = AutoModelForSequenceClassification.from_pretrained('models/sentiment_model')

embedder = SentenceTransformer('all-MiniLM-L6-v2')
faiss_index = faiss.read_index('models/faiss_index.bin')

with open('models/documents.json', 'r', encoding='utf-8') as f:
    documents = json.load(f)

t5_tokenizer = T5TokenizerFast.from_pretrained('models/flan_t5_tokenizer')
t5_model = T5ForConditionalGeneration.from_pretrained('models/flan_t5_model')

print("✓ Models loaded")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 540.94it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 282/282 [00:00<00:00, 464.76it/s, Materializing param=shared.weight]                                                       
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✓ Models loaded


## Load Citizen Charter Data

In [3]:
with open('../2. dataset preparation/citizen_charter_services.json', 'r', encoding='utf-8') as f:
    charter_data = json.load(f)

print(f"Loaded {len(charter_data)} offices")
print(f"Total services: {sum(len(office['services']) for office in charter_data)}")

Loaded 2 offices
Total services: 2


## Create Test Questions

In [4]:
test_questions = [
    "How do I get a business permit?",
    "What are the requirements for marriage license?",
    "Where can I get a tax declaration?",
    "How to apply for building permit?",
    "What documents do I need for birth certificate?",
    "How can I get anti-rabies vaccination for my dog?",
    "What is the process for getting a mayor's permit?",
    "How do I register a death certificate?",
    "What are the requirements for scholarship grant?",
    "How to get technical assistance for farming?"
]

print(f"Created {len(test_questions)} test questions")

Created 10 test questions


## Test RAG Retrieval Accuracy

In [5]:
def retrieve_context(query, k=3):
    query_embedding = embedder.encode([query])
    distances, indices = faiss_index.search(query_embedding.astype('float32'), k)
    return [documents[i] for i in indices[0]], distances[0]

print("Testing RAG Retrieval...\n")
for question in test_questions:
    context, distances = retrieve_context(question, k=3)
    print(f"Q: {question}")
    print(f"Top 3 Retrieved Services:")
    for i, (doc, dist) in enumerate(zip(context, distances)):
        print(f"  {i+1}. {doc['title'][:80]}... (distance: {dist:.2f})")
    print()

Testing RAG Retrieval...

Q: How do I get a business permit?
Top 3 Retrieved Services:
  1. BUSINESS PERMIT NEW AND RENEWAL ONLINE APPLICATION... (distance: 0.97)
  2. Issuance of certificate of inspection for new and renewal of business permits (P... (distance: 1.02)
  3. ISSUANCE OF ZONING CERTIFICATE... (distance: 1.11)

Q: What are the requirements for marriage license?
Top 3 Retrieved Services:
  1. ISSUANCE OF MARRIAGE LICENSE... (distance: 0.63)
  2. PROVISION OF MARRIAGE ADMINISTRATION/ SOLEMNIZATION... (distance: 0.84)
  3. REGISTERING MARRIAGE CERTIFICATES... (distance: 0.92)

Q: Where can I get a tax declaration?
Top 3 Retrieved Services:
  1. ISSUANCE OF CERTIFIED TRUE COPY OF TAX DECLARATION... (distance: 0.91)
  2. ISSUANCE OF TAX DECLARATION OF REAL PROPERTY FOR SUBDIVIDED PARCEL OF LAND (with... (distance: 0.93)
  3. ISSUANCE OF TAX DECLARATION OF REAL PROPERTY FOR CONSOLIDATED PARCEL OF LAND (wi... (distance: 0.95)

Q: How to apply for building permit?
Top 3 Retrieved 

## Test Time Waster Detection

In [6]:
def detect_time_waster(text):
    vec = tfidf.transform([text.strip().lower()])
    return tw_classifier.predict(vec)[0] == 1

time_waster_tests = [
    ("hello", True),
    ("test", True),
    ("???", True),
    ("How do I get a business permit?", False),
    ("What are the requirements?", False),
]

print("Testing Time Waster Detection...\n")
correct = 0
for text, expected in time_waster_tests:
    result = detect_time_waster(text)
    status = "✓" if result == expected else "✗"
    print(f"{status} '{text}' -> {result} (expected: {expected})")
    if result == expected:
        correct += 1

print(f"\nAccuracy: {correct}/{len(time_waster_tests)} ({100*correct/len(time_waster_tests):.1f}%)")

Testing Time Waster Detection...

✓ 'hello' -> True (expected: True)
✓ 'test' -> True (expected: True)
✓ '???' -> True (expected: True)
✓ 'How do I get a business permit?' -> False (expected: False)
✓ 'What are the requirements?' -> False (expected: False)

Accuracy: 5/5 (100.0%)


## Test Sentiment Analysis

In [7]:
def analyze_sentiment(text):
    inputs = sentiment_tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    outputs = sentiment_model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    return "positive" if probs[0][1] > probs[0][0] else "negative"

sentiment_tests = [
    "I'm very happy with the service!",
    "This is frustrating and taking too long",
    "Thank you for your help",
    "I'm angry about this process",
    "The staff was very helpful",
]

print("Testing Sentiment Analysis...\n")
for text in sentiment_tests:
    sentiment = analyze_sentiment(text)
    print(f"{text}")
    print(f"  → Sentiment: {sentiment}\n")

Testing Sentiment Analysis...



I'm very happy with the service!
  → Sentiment: positive

This is frustrating and taking too long
  → Sentiment: negative

Thank you for your help
  → Sentiment: positive

I'm angry about this process
  → Sentiment: negative

The staff was very helpful
  → Sentiment: positive



## Test Full Pipeline

In [8]:
def generate_answer(query, context, sentiment):
    context_text = "\n".join([f"Service: {c['title']}. Office: {c['office']}" for c in context])
    
    if sentiment == "negative":
        tone = "Respond professionally with empathy and comfort. "
    else:
        tone = "Respond professionally and helpfully. "
    
    prompt = f"{tone}Context: {context_text}\n\nQuestion: {query}\n\nAnswer:"
    
    inputs = t5_tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    outputs = t5_model.generate(**inputs, max_length=150)
    return t5_tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Testing Full Pipeline...\n")
for question in test_questions[:5]:  # Test first 5
    print(f"Q: {question}")
    
    sentiment = analyze_sentiment(question)
    context, _ = retrieve_context(question)
    answer = generate_answer(question, context, sentiment)
    
    print(f"Sentiment: {sentiment}")
    print(f"Answer: {answer}")
    print("-" * 80)
    print()

Testing Full Pipeline...

Q: How do I get a business permit?
Sentiment: negative
Answer: BUSINESS PERMIT NEW AND RENEWAL ONLINE APPLICATION
--------------------------------------------------------------------------------

Q: What are the requirements for marriage license?
Sentiment: negative
Answer: a marriage license must be issued by a municipal civil registry office
--------------------------------------------------------------------------------

Q: Where can I get a tax declaration?
Sentiment: negative
Answer: Office: Municipal Assessor's Office
--------------------------------------------------------------------------------

Q: How to apply for building permit?
Sentiment: negative
Answer: Office: Municipal Planning and Development Office Service: ISSUANCE OF ZONING CERTIFICATE. Office: Municipal Planning and Development Office Service: ISSUANCE OF LOCATIONAL CLEARANCE (For Building Permit of Environmentally Critical Projects). Office: Municipal Engineering Office Service: ISSUANCE

## Evaluation Summary

In [9]:
print("="*80)
print("EVALUATION SUMMARY")
print("="*80)
print(f"✓ RAG Retrieval: Working - retrieves top 3 relevant services")
print(f"✓ Time Waster Detection: Tested on {len(time_waster_tests)} cases")
print(f"✓ Sentiment Analysis: Detects positive/negative sentiment")
print(f"✓ Answer Generation: FLAN-T5 generates responses based on context")
print(f"\nTotal Services in Database: {len(documents)}")
print(f"Total Offices: {len(charter_data)}")

EVALUATION SUMMARY
✓ RAG Retrieval: Working - retrieves top 3 relevant services
✓ Time Waster Detection: Tested on 5 cases
✓ Sentiment Analysis: Detects positive/negative sentiment
✓ Answer Generation: FLAN-T5 generates responses based on context

Total Services in Database: 109
Total Offices: 2
